### 0. Setup

In [12]:
# Librerias - load data

import pandas as pd
import numpy as np

# Load raw datasets (6 countries version)
df = pd.read_csv("../Data/ofertas_adzuna_data_raw_v4.csv")
df_hist = pd.read_csv("../Data/historico_salarios_data_raw_v3.csv")

# Check shape of both datasets
print("OFFERS")
print("Shape:", df.shape)
print("\nHISTORICAL")
print("Shape:", df_hist.shape)

df.head()


OFFERS
Shape: (9289, 12)

HISTORICAL
Shape: (288, 4)


,title,description,category,category_tag,salary_min,salary_max,salary_is_predicted,location,company,contract_type,created,pais
0,Customer Data & Activation Engineer,¿Quieres formar parte de una empresa líder en ...,Trabajos en informática,it-jobs,NaN,NaN,0,España,Cognodata,NaN,2026-05-24T22:18:26Z,es
1,Desarrollador de PHP - (Girona),Empresas: Eurostars Hotel Company (Grupo Hotus...,Trabajos en informática,it-jobs,NaN,NaN,0,Girona,Eurostars Hotel Company,NaN,2026-05-18T12:36:24Z,es
2,Technical Business Analyst,Here at CORUS we are looking for a Technical B...,Trabajos en informática,it-jobs,NaN,NaN,0,Valencia,CORUS Consulting,NaN,2026-05-15T11:21:06Z,es
3,Ukrainian Language Specialist - Freelance AI T...,Are you a Ukrainian language expert eager to s...,Trabajos en informática,it-jobs,16640.0,135200.0,0,España,Invisible Agency,NaN,2025-10-31T18:50:50Z,es
4,Technical Account Manager,Technical Account Manager Location: Spain / Ma...,Trabajos en informática,it-jobs,NaN,NaN,0,España,Maisa AI,NaN,2026-05-14T00:13:34Z,es


**Observación inicial de calidad de los datos**

Antes de limpiar, revisamos qué problemas tiene el dataset.

Una primera observación al mirar las ofertas: algunas filas **no tienen salario** (`salary_min` y `salary_max` vacíos) y además `salary_is_predicted = 0`, lo que indica que la API no lo estimó. Esto parece variar según el país.

En las siguientes celdas se comproborá lo siguiente:
- Tipos de cada columna y valores no nulos
- Cuántos nulos hay por columna
- Filas duplicadas
- Cuántas ofertas no tienen salario, repartidas por país (clave, porque el salario es la variable principal)

### 1. Initial data check

In [13]:
# Column types and non-null counts
df.info()

# Null values per column
print("\nNULLS PER COLUMN")
print(df.isnull().sum())

# Duplicated rows
print("\n DUPLICATES")
print("Duplicated rows:", df.duplicated().sum())

# Salary nulls by country 
print("\nROWS WITHOUT SALARY, BY COUNTRY")
print(df[df["salary_min"].isnull()].groupby("pais").size())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9289 entries, 0 to 9288
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   title                9289 non-null   object 
 1   description          9289 non-null   object 
 2   category             9289 non-null   object 
 3   category_tag         9289 non-null   object 
 4   salary_min           4571 non-null   float64
 5   salary_max           4564 non-null   float64
 6   salary_is_predicted  9289 non-null   int64  
 7   location             9289 non-null   object 
 8   company              9000 non-null   object 
 9   contract_type        2202 non-null   object 
 10  created              9289 non-null   object 
 11  pais                 9289 non-null   object 
dtypes: float64(2), int64(1), object(9)
memory usage: 871.0+ KB

NULLS PER COLUMN
title                     0
description               0
category                  0
category_tag              0

In [14]:
# Rows with salary, by country
print("ROWS WITH SALARY, BY COUNTRY")
print(df[df["salary_min"].notnull()].groupby("pais").size())

ROWS WITH SALARY, BY COUNTRY
pais
de      50
es     242
fr     635
gb    1600
nl     444
us    1600
dtype: int64


In [15]:
# Salary type: real (0) vs predicted (1), only for rows that have salary
print("SALARY TYPE (rows with salary)")
print(df[df["salary_min"].notnull()]["salary_is_predicted"].value_counts())

SALARY TYPE (rows with salary)
salary_is_predicted
0    2370
1    2201
Name: count, dtype: int64


**Comentario**

Tras revisar el dataset de ofertas (9.289 filas, 6 países), se detectan los siguientes problemas en los datos:

**1. Salarios vacíos (problema principal)**
- Solo 4.571 ofertas (49%) tienen salario; 4.718 lo tienen vacío.
- De las que tienen salario, 2.370 son reales y 2.201 estimadas por la API.
- La cobertura varía mucho por país: UK y USA tienen cobertura total (1.600 cada uno) y Francia buena (635), mientras que Alemania apenas tiene 50 ofertas con salario.

**2. `contract_type` muy incompleto**
- 7.087 nulos de 9.289 (76%). Columna casi vacía.

**3. `company` con algunos nulos**
- 289 nulos (3%)

**4. Filas duplicadas**
- 33 duplicados exactos (ofertas repetidas con el mismo título y descripción).

**5. Categorías en varios idiomas**
- La variable `category` viene en el idioma de cada país ("IT Jobs", "IT-Stellen", "Trabajos en informática"). Para unificar sectores usamos `category_tag`, que es un código uniforme en todos los países (`it-jobs`, `retail-jobs`, etc.).

**6. Salarios en distintas monedas**
- Los salarios vienen en la moneda local: euros (España, Francia, Alemania, Países Bajos), libras (Reino Unido) y dólares (USA). Para poder compararlos se convierten todos a euros usando los tipos de cambio de referencia del Banco Central Europeo (BCE) a fecha 3 de junio de 2026.

**Decisiones tomadas**
- **Dos enfoques de análisis** según la calidad del dato:
  - *Análisis salarial*: solo con las ofertas que tienen salario y un valor fiable (≥10.000 €). Alemania tendrá baja cobertura, lo que se indicará como limitación.
  - *Análisis de volumen/demanda*: con todas las ofertas, ya que el número de ofertas por país y sector es un dato válido aunque falte el salario.
- Se recogió también el campo `description` para poder extraer información adicional (modalidad remoto/híbrido, nivel del puesto).

### 2. Remove duplicates

In [16]:
# antes de borrar, miramos todas las filas duplicadas
duplicates = df[df.duplicated(keep=False)]
print("Total filas duplicadas:", len(duplicates))

# ordenar por título para ver juntas las que son iguales y compararlas
duplicates_sorted = duplicates.sort_values("title")
duplicates_sorted[["title", "company", "pais", "salary_min", "category_tag"]].head(20)

Total filas duplicadas: 66


,title,company,pais,salary_min,category_tag
472,Asesor/a Comercial Mutualidad Previsión,"hna Mutualidad de los Arquitectos, Arquitectos...",es,NaN,accounting-finance-jobs
665,Asesor/a Comercial Mutualidad Previsión,"hna Mutualidad de los Arquitectos, Arquitectos...",es,NaN,accounting-finance-jobs
462,Asesor/a Fiscal - Tax Advisor - Cliente Final.,NaN,es,30000.0,accounting-finance-jobs
754,Asesor/a Fiscal - Tax Advisor - Cliente Final.,NaN,es,30000.0,accounting-finance-jobs
735,Beca Departamento Control De Gestión Agrícola ...,Florette,es,NaN,accounting-finance-jobs
488,Beca Departamento Control De Gestión Agrícola ...,Florette,es,NaN,accounting-finance-jobs
712,COMMERCIAL GRADUATE EXPERT (English-German),Saica,es,NaN,accounting-finance-jobs
446,COMMERCIAL GRADUATE EXPERT (English-German),Saica,es,NaN,accounting-finance-jobs
751,Coordinador/A Modelos Financieros,Enagás,es,NaN,accounting-finance-jobs
594,Coordinador/A Modelos Financieros,Enagás,es,NaN,accounting-finance-jobs


In [17]:
# Remove duplicate rows
filas_antes = len(df)
df = df.drop_duplicates()
filas_despues = len(df)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Eliminadas:", filas_antes - filas_despues)

Filas antes: 9289
Filas después: 9256
Eliminadas: 33


### 3. Standarization

In [18]:
# Renombramos los valores del category_tag a nombres más legibles
df["category_tag"] = df["category_tag"].replace({
    "it-jobs": "IT",
    "accounting-finance-jobs": "Finance",
    "retail-jobs": "Retail",
    "logistics-warehouse-jobs": "Logistics"
})

# Comprobamos 
print(df["category_tag"].value_counts())

category_tag
IT           2400
Logistics    2400
Finance      2368
Retail       2088
Name: count, dtype: int64


In [19]:
# Renombramos la columna a sector
df = df.rename(columns={"category_tag": "sector"})

# Y también quitamos la columna 'category' original (idiomas mezclados, ya no la usamos)
df = df.drop(columns=["category"])

print(list(df.columns))

['title', 'description', 'sector', 'salary_min', 'salary_max', 'salary_is_predicted', 'location', 'company', 'contract_type', 'created', 'pais']


### 4. Contract_type column

In [20]:
# See the values of contract_type (ignoring nulls)
print(df["contract_type"].value_counts(dropna=False))

contract_type
NaN          7054
permanent    1185
contract     1017
Name: count, dtype: int64


In [21]:
# Drop contract_type (hight % empty, not useful for analysis)

df = df.drop(columns=["contract_type"])

print(list(df.columns))

['title', 'description', 'sector', 'salary_min', 'salary_max', 'salary_is_predicted', 'location', 'company', 'created', 'pais']


### 5. Extraer de 'description': nivel, idioma, modalidad y herramientas IT

In [22]:
# 5.1 Nivel
# Extraemos el nivel del puesto buscando en el título Y en la descripción
def detectar_nivel(fila):
    # Juntamos título y descripción en un solo texto, en minúsculas
    texto = (str(fila["title"]) + " " + str(fila["description"])).lower()
    
    # Variantes de Senior
    if "senior" in texto or "sr." in texto or "sr " in texto or "lead" in texto:
        return "Senior"
    # Variantes de Junior
    elif "junior" in texto or "jr." in texto or "jr " in texto or "intern" in texto or "becario" in texto:
        return "Junior"
    else:
        return "Not specified"

# Aplicamos la función a cada fila
df["nivel"] = df.apply(detectar_nivel, axis=1)

print(df["nivel"].value_counts())

nivel
Not specified    6327
Senior           2053
Junior            876
Name: count, dtype: int64


In [23]:
# 5.2 Idioma
# Detectamos los idiomas mencionados en el título y la descripción
def detectar_idioma(fila):
    texto = (str(fila["title"]) + " " + str(fila["description"])).lower()
    
    idiomas = []
    
    if "english" in texto or "inglés" in texto or "ingles" in texto:
        idiomas.append("English")
    if "español" in texto or "spanish" in texto or "castellano" in texto:
        idiomas.append("Spanish")
    if "deutsch" in texto or "german" in texto or "alemán" in texto:
        idiomas.append("German")
    if "français" in texto or "french" in texto or "francés" in texto:
        idiomas.append("French")
    if "nederlands" in texto or "dutch" in texto or "holandés" in texto:
        idiomas.append("Dutch")
    
    # Si no menciona ningún idioma, lo marcamos
    if len(idiomas) == 0:
        return "Not mentioned"
    
    # Unimos los idiomas encontrados en un texto separado por comas
    return ", ".join(idiomas)

# Aplicamos la función a cada fila
df["idiomas"] = df.apply(detectar_idioma, axis=1)

# Comprobamos el resultado
print(df["idiomas"].value_counts().head(15))

idiomas
Not mentioned                     8614
German                             385
English                             88
French                              76
Dutch                               54
Spanish                             26
English, Spanish                     5
English, French                      2
English, Dutch                       2
English, German                      1
English, Spanish, French             1
Spanish, German, French, Dutch       1
German, Dutch                        1
Name: count, dtype: int64


In [24]:
# Quitamos la columna 'idiomas' va a próximos pasos
df = df.drop(columns=["idiomas"])

print("Columnas ahora:", list(df.columns))

Columnas ahora: ['title', 'description', 'sector', 'salary_min', 'salary_max', 'salary_is_predicted', 'location', 'company', 'created', 'pais', 'nivel']


**Idioma: descartado del análisis (movido a próximos pasos)**

Se intentó extraer el idioma requerido en cada oferta buscando menciones explícitas ("english", "deutsch", etc.) en el título y la descripción. Sin embargo, el resultado no es fiable: el 92% de las ofertas quedaban como "no menciona", porque las ofertas raramente indican de forma explícita el idioma requerido (normalmente el idioma se refleja en cómo está escrito el anuncio, no en una frase concreta).

Detectar el idioma real exigido es un problema complejo:
- El idioma en que está **escrito** el anuncio no siempre coincide con el idioma **requerido** para el puesto (muchas empresas publican en inglés aunque el trabajo sea en el idioma local, y viceversa).
- Hacerlo bien requeriría técnicas de procesamiento de lenguaje natural (NLP) más avanzadas.

Por eso se elimina la columna `idiomas` y este análisis se deja como **trabajo futuro**, en lugar de incluir un dato poco fiable que podría llevar a conclusiones erróneas.

In [25]:
# 5.3 Modalidad de trabajo

# Detectamos la modalidad de trabajo en el título y la descripción
def detectar_modalidad(fila):
    texto = (str(fila["title"]) + " " + str(fila["description"])).lower()
    
    # Remoto (inglés, español, francés, alemán, neerlandés)
    if ("remote" in texto or "remoto" in texto or "teletrabajo" in texto 
        or "télétravail" in texto or "fernarbeit" in texto or "thuiswerken" in texto):
        return "Remote"
    # Híbrido
    elif ("hybrid" in texto or "híbrido" in texto or "hibrido" in texto 
          or "hybride" in texto or "hybrid" in texto):
        return "Hybrid"
    # Presencial
    elif ("on-site" in texto or "onsite" in texto or "presencial" in texto 
          or "on site" in texto or "vor ort" in texto or "op locatie" in texto):
        return "On-site"
    else:
        return "Not specified"

# Aplicamos la función a cada fila
df["modalidad"] = df.apply(detectar_modalidad, axis=1)

print(df["modalidad"].value_counts())

modalidad
Not specified    8575
Hybrid            265
Remote            249
On-site           167
Name: count, dtype: int64


**Modalidad de trabajo (remoto / híbrido / presencial)**

Se extrae la modalidad de trabajo buscando palabras clave en el título y la descripción, en los idiomas de los 6 países (ej. "remote", "teletrabajo", "hybrid", "presencial", "fernarbeit", "thuiswerken").

La mayoría de ofertas  no especifican la modalidad de forma explícita, por lo que quedan como "Not specified". Sin embargo, entre las que sí la indican, el reparto es informativo (híbrido, remoto y presencial en proporciones similares).

**Decisión:** se mantiene la variable `modalidad`, pero los análisis sobre ella se harán solo sobre las ofertas que la especifican, indicando claramente que es una submuestra. Aporta un insight sobre flexibilidad laboral más allá del salario.

In [26]:
# 5.4 herramientas IT

# Detectamos qué herramientas se mencionan en el título y la descripción
def detectar_herramientas(fila):
    texto = (str(fila["title"]) + " " + str(fila["description"])).lower()
    
    herramientas = []
    
    if "python" in texto:
        herramientas.append("Python")
    if "sql" in texto:
        herramientas.append("SQL")
    if "tableau" in texto:
        herramientas.append("Tableau")
    if "power bi" in texto or "powerbi" in texto:
        herramientas.append("Power BI")
    if "excel" in texto:
        herramientas.append("Excel")
    
    if len(herramientas) == 0:
        return "None mentioned"
    
    return ", ".join(herramientas)

# Aplicamos la función a cada fila
df["herramientas"] = df.apply(detectar_herramientas, axis=1)

# Comprobamos el reparto general
print(df["herramientas"].value_counts().head(15))


herramientas
None mentioned               8683
Excel                         470
Python                         53
SQL                            23
Power BI                       10
Tableau                         9
SQL, Power BI                   3
Python, SQL                     2
Python, Tableau, Power BI       1
Power BI, Excel                 1
SQL, Excel                      1
Name: count, dtype: int64


In [27]:
df = df.drop(columns=["herramientas"])

**Herramientas / tecnologías: descartado (movido a próximos pasos)**

Se intentó detectar las herramientas mencionadas en las ofertas (Python, SQL, Tableau, Power BI, Excel) buscando palabras clave en el título y la descripción. Sin embargo, el 94% de las ofertas no mencionan ninguna herramienta, y el reparto resultante no es coherente (Excel aparece mucho más que Python o SQL, lo cual es improbable en un dataset con ofertas IT).

La causa más probable es que la descripción que proporciona la API de Adzuna viene **resumida** (recortada con "..."), por lo que la sección de requisitos técnicos, donde suelen aparecer las herramientas, queda cortada.

**Decisión:** se descarta este análisis por falta de datos fiables y se deja como **línea de trabajo futura**: analizar las herramientas más demandadas requeriría el texto completo de las ofertas.

### 6. 'created' column

In [28]:
# Convertimos 'created' a formato fecha para poder analizarla
df["created"] = pd.to_datetime(df["created"])

# Vemos el rango de fechas (la más antigua y la más reciente)
print("Fecha más antigua:", df["created"].min())
print("Fecha más reciente:", df["created"].max())

# Cuántas ofertas hay por mes
df["mes"] = df["created"].dt.to_period("M")
print("\nOfertas por mes:")
print(df["mes"].value_counts().sort_index())

Fecha más antigua: 2017-03-27 17:00:46+00:00
Fecha más reciente: 2026-06-04 10:39:18+00:00

Ofertas por mes:
mes
2017-03       1
2019-09       1
2019-10       8
2022-01       1
2022-11      44
2023-01       7
2023-02      67
2023-03       1
2023-04       1
2023-06      31
2023-10       5
2023-11       4
2023-12       5
2024-01       2
2024-02       4
2024-03       6
2024-04       2
2024-05       4
2024-06       3
2024-07      20
2024-08       6
2024-09       7
2024-10       7
2024-11       7
2024-12       6
2025-01       3
2025-02      13
2025-03      34
2025-04       8
2025-05      24
2025-06      17
2025-07      52
2025-08      36
2025-09      86
2025-10     108
2025-11     145
2025-12     153
2026-01     143
2026-02     318
2026-03     394
2026-04     924
2026-05    4808
2026-06    1740
Freq: M, Name: count, dtype: int64


/var/folders/vq/dm0z_g7d7gnc_rx3fbh36mtm0000gn/T/ipykernel_88708/3626468539.py:9: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["mes"] = df["created"].dt.to_period("M")


In [29]:
# Quitamos 'created' y 'mes': las fechas están concentradas en mayo 2026,
# no sirven para análisis temporal. Para tendencias usamos el histórico.
df = df.drop(columns=["created", "mes"])

print("Columnas ahora:", list(df.columns))

Columnas ahora: ['title', 'description', 'sector', 'salary_min', 'salary_max', 'salary_is_predicted', 'location', 'company', 'pais', 'nivel', 'modalidad']


### 7. Check currency of salaries

In [36]:
# Convertimos los salarios a euros según la moneda local de cada país
# Tipos de cambio de referencia del BCE (3 junio 2026)

def convertir_a_euros(fila, columna):
    salario = fila[columna]
    pais = fila["pais"]
    
    # Si no hay salario, lo dejamos como está (NaN)
    if pd.isnull(salario):
        return salario
    
    # UK está en libras → dividimos entre 0.8637
    if pais == "gb":
        return salario / 0.8637
    # USA está en dólares → dividimos entre 1.1614
    elif pais == "us":
        return salario / 1.1614
    # El resto ya está en euros
    else:
        return salario

# Aplicamos la conversión a salary_min y salary_max
df["salary_min"] = df.apply(lambda fila: convertir_a_euros(fila, "salary_min"), axis=1)
df["salary_max"] = df.apply(lambda fila: convertir_a_euros(fila, "salary_max"), axis=1)

print("Salarios convertidos a euros")

# Comprobamos las medias por país (ahora todas en euros, comparables)
print("\nSalario medio por país (en euros):")
print(df.groupby("pais")["salary_min"].mean().round(0))

Salarios convertidos a euros

Salario medio por país (en euros):
pais
de    28300.0
es    37041.0
fr    25822.0
gb    49245.0
nl    33353.0
us    74850.0
Name: salary_min, dtype: float64


**Conversión de salarios a euros**

Los salarios vienen en la moneda local de cada país: euros (España, Francia, Alemania, Países Bajos), libras (Reino Unido) y dólares (USA). Para poder compararlos entre países, se convierten todos a euros.

Se usan los tipos de cambio de referencia oficiales del Banco Central Europeo (BCE) a fecha 3 de junio de 2026:
- 1 EUR = 0,8637 GBP → los salarios de UK se dividen entre 0,8637
- 1 EUR = 1,1614 USD → los salarios de USA se dividen entre 1,1614

Tras la conversión, los salarios medios por país (en euros) muestran que USA y Reino Unido encabezan la tabla, seguidos de España. La conversión es imprescindible y sin ella, la comparación entre países sería incorrecta (por ejemplo, los salarios de UK estaban infravalorados al estar expresados en libras). Por otro lado, Alemania y Francia bajos y como se había mencionado ante, los datos de Alemania es poco fiable (50 datos).

In [37]:
# Look at the salary range to find outliers
print("Lowest salaries")
print(df[df["salary_min"].notnull()]["salary_min"].sort_values().head(10))

print("\n Highest salaries ")
print(df[df["salary_min"].notnull()]["salary_min"].sort_values().tail(10))

Lowest salaries
1584    0.0
2663    0.0
2668    0.0
2678    0.0
2684    0.0
2691    0.0
2696    0.0
2701    0.0
2704    0.0
2721    0.0
Name: salary_min, dtype: float64

 Highest salaries 
8460    253025.762011
8097    254937.575340
8325    258295.677630
7972    264470.415016
7981    266656.294128
7999    364918.012743
8226    377555.992767
1916    437651.962487
3481    444000.000000
3438    600000.000000
Name: salary_min, dtype: float64


In [39]:
# Vemos cuántas ofertas quedan fuera por cada límite
print("Salarios = 0:", len(df[df["salary_min"] == 0]))
print("Salarios entre 1 y 10.000:", len(df[(df["salary_min"] > 0) & (df["salary_min"] < 10000)]))
print("Salarios > 300.000:", len(df[df["salary_min"] > 300000]))

Salarios = 0: 55
Salarios entre 1 y 10.000: 364
Salarios > 300.000: 5


In [41]:
# Creamos el dataset salarial: salarios fiables entre 10.000 y 300.000 euros
df_salarios = df[(df["salary_min"] >= 10000) & (df["salary_min"] <= 300000)].copy()

print("Dataset completo (volumen):", len(df), "filas")
print("Dataset salarial:", len(df_salarios), "filas")
print("\nOfertas con salario fiable por país:")
print(df_salarios.groupby("pais").size())

Dataset completo (volumen): 9256 filas
Dataset salarial: 4138 filas

Ofertas con salario fiable por país:
pais
de      36
es     215
fr     432
gb    1510
nl     353
us    1592
dtype: int64


**Limpieza de salarios**

La variable salario es el eje principal del análisis, pero presentaba varios problemas de calidad:

- **Valores vacíos:** el 51% de las ofertas no tienen salario. Se mantienen en el dataset de volumen, pero se excluyen del análisis salarial.
- **Valores demasiado bajos:** había salarios de 0 (55 ofertas) y valores entre 1 y 10.000 (364 ofertas), que corresponden a salarios por hora o mensuales mal informados, no anuales. Se aplica un **suelo de 10.000 €**.
- **Valores demasiado altos:** algunos salarios eran claramente errores (5 ofertas por encima de 300.000 €, ej. un puesto de "Assistant" con 437.000 €, o 600.000 €). Se aplica un **techo de 300.000 €**, que deja pasar sueldos directivos reales (incluidos los salarios altos de USA) pero descarta los disparatados.

**Resultado:** se crean dos datasets:
- `ofertas_clean` (9.256 filas): todas las ofertas, para el análisis de volumen/demanda.
- `ofertas_salarios_clean` (4.138 filas): solo ofertas con salario fiable (entre 10.000 y 300.000 €), para el análisis salarial.

Este doble enfoque permite aprovechar todos los datos según su calidad, sin perder información ni forzar valores poco fiables

### 7. Limpieza historico salarios

In [43]:
# check inicial
print("Shape:", df_hist.shape)
print("\nPaíses:", df_hist["pais"].unique())
df_hist.head()

Shape: (288, 4)

Países: ['es' 'gb' 'fr' 'de' 'nl' 'us']


,pais,categoria,mes,salario_medio
0,es,it-jobs,2025-10,85169.46
1,es,it-jobs,2026-05,46960.80
2,es,it-jobs,2025-09,85546.96
3,es,it-jobs,2026-03,45262.46
4,es,it-jobs,2025-07,84843.87


In [44]:
# Renombramos las categorías a nombres legibles (igual que en las ofertas)
df_hist["categoria"] = df_hist["categoria"].replace({
    "it-jobs": "IT",
    "accounting-finance-jobs": "Finance",
    "retail-jobs": "Retail",
    "logistics-warehouse-jobs": "Logistics"
})

# Convertimos el salario medio a euros (UK y USA están en moneda local)
def convertir_historico(fila):
    salario = fila["salario_medio"]
    pais = fila["pais"]
    
    if pais == "gb":
        return salario / 0.8637      # libras a euros
    elif pais == "us":
        return salario / 1.1614      # dólares a euros
    else:
        return salario               # ya está en euros

df_hist["salario_medio"] = df_hist.apply(convertir_historico, axis=1)

# 3. Renombramos la columna y ordenamos
df_hist = df_hist.rename(columns={"categoria": "sector"})
df_hist = df_hist.sort_values(["pais", "sector", "mes"])

print(df_hist.head(12))

    pais   sector      mes  salario_medio
166   de  Finance  2025-06       60720.83
162   de  Finance  2025-07       60378.43
163   de  Finance  2025-08       56653.43
160   de  Finance  2025-09       57828.32
158   de  Finance  2025-10       61521.47
161   de  Finance  2025-11       61466.38
167   de  Finance  2025-12       61123.69
164   de  Finance  2026-01       61381.78
156   de  Finance  2026-02       60358.03
159   de  Finance  2026-03       60108.20
157   de  Finance  2026-04       54555.95
165   de  Finance  2026-05       57483.08


In [45]:
# Verificamos que UK y USA se convirtieron a euros
print("UK (estaba en libras)")
print(df_hist[df_hist["pais"] == "gb"].head(3))

print("\nUSA (estaba en dolares)")
print(df_hist[df_hist["pais"] == "us"].head(3))

UK (estaba en libras)
   pais   sector      mes  salario_medio
70   gb  Finance  2025-06   56614.368415
62   gb  Finance  2025-07   56047.991201
68   gb  Finance  2025-08   56641.194859

USA (estaba en dolares)
    pais   sector      mes  salario_medio
258   us  Finance  2025-06   85075.133460
259   us  Finance  2025-07   85257.611503
263   us  Finance  2025-08   84966.273463


In [47]:
# Redondeamos los salarios a 2 decimales (quedaron con decimales largos tras la conversion)
df["salary_min"] = df["salary_min"].round(2)
df["salary_max"] = df["salary_max"].round(2)

df_salarios["salary_min"] = df_salarios["salary_min"].round(2)
df_salarios["salary_max"] = df_salarios["salary_max"].round(2)

df_hist["salario_medio"] = df_hist["salario_medio"].round(2)

### 8. Datasets limpios finales

In [48]:
# Guardamos 6 paises, salarios en euros
df.to_csv("../Data/ofertas_clean.csv", index=False, encoding="utf-8")
df_salarios.to_csv("../Data/ofertas_salarios_clean.csv", index=False, encoding="utf-8")
df_hist.to_csv("../Data/historico_clean.csv", index=False, encoding="utf-8")

print("ofertas_clean.csv:", len(df), "filas")
print("ofertas_salarios_clean.csv:", len(df_salarios), "filas")
print("historico_clean.csv:", len(df_hist), "filas")

ofertas_clean.csv: 9256 filas
ofertas_salarios_clean.csv: 4138 filas
historico_clean.csv: 288 filas
